# Train the DQN autopilot on Google Colab

Training is fully headless (no pygame, no window). Only numpy and PyTorch are used, and both are preinstalled on Colab.

1. **Runtime → Change runtime type → T4 GPU**
2. Upload `vertical-shmup.zip` to the top of your Google Drive (`MyDrive/`).
3. Run the cells in order. Checkpoints are written to `MyDrive/shmup/`, so a disconnect loses nothing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/shmup'
!mkdir -p $DRIVE

In [ ]:
!rm -rf /content/vertical-shmup
!unzip -q -o /content/drive/MyDrive/vertical-shmup.zip -d /content
%cd /content/vertical-shmup
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, numpy; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## Train
Logs are appended to `MyDrive/shmup/train.log`. `--total-steps 3000000` is a good first run; the default GPU profile is 8M.

In [ ]:
!python -u scripts/train_gpu.py --total-steps 3000000 --save-path $DRIVE/dqn.pt 2>&1 | tee -a $DRIVE/train.log

## Resume after a disconnect
Re-run the first two cells, then this one. It continues from the last evaluation checkpoint.

In [ ]:
!python -u scripts/train_gpu.py --total-steps 3000000 --save-path $DRIVE/dqn.pt --resume $DRIVE/dqn_last.pt 2>&1 | tee -a $DRIVE/train.log

## Evaluate and export for the web demo

In [ ]:
!python scripts/train_gpu.py --eval-only --save-path $DRIVE/dqn.pt
!python scripts/export_web.py --model $DRIVE/dqn.pt
!cp web/model/dqn_model.js $DRIVE/

In [ ]:
from google.colab import files
files.download(DRIVE + '/dqn.pt')
files.download(DRIVE + '/dqn_model.js')